# Bài 12
Đây là notebook chứa mã nguồn đầy đủ của bài 12.

In [ ]:
from ipynb.fs.full.notebooks.bai01_notebook import SCENARIOS

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import numpy as np
import pandas as pd
import pulp
import pyomo.environ as pyo
from scipy.optimize import linprog, minimize, milp, LinearConstraint, Bounds
from pymoo.core.problem import ElementwiseProblem
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.optimize import minimize as pymoo_minimize

from src.data_loader import get_data


In [ ]:
def solve_m1_macro(budget, scenario_cfg):
    """
    Dự báo GDP qua hàm sản xuất Cobb-Douglas.
    
    Args:
        budget: Ngân sách đầu tư
        scenario_cfg: Dictionary cấu hình kịch bản (chứa tfp_growth, alloc_weights)
        
    Returns:
        dict: Chứa list 'years' và 'gdp'
    """
    a, b, g, d, t = 0.33, 0.42, 0.10, 0.08, 0.07
    s = a + b + g + d + t
    a, b, g, d, t = a/s, b/s, g/s, d/s, t/s

    K0, L0, D0, AI0, H0 = 25900, 53.4, 19.5, 80.1, 29.2
    tfpg = scenario_cfg['tfp_growth']
    A0 = 1.0

    years = list(range(2026, 2031))
    gdp   = []
    K, L, D, AI, H, A = K0, L0, D0, AI0, H0, A0
    
    for i in range(5):
        g_val = A * (K**a) * (L**b) * (D**g) * (AI**d) * (H**t)
        gdp.append(round(float(g_val / 1000), 2))  # nghìn tỷ
        K  = K  * 1.06
        L  = L  * 1.005
        D  = D  + 1.5
        AI = min(AI * 1.08, 100)
        H  = H  + 0.5
        A  = A  * (1 + tfpg)
        
    return {'years': years, 'gdp': gdp}

In [ ]:
def solve_m2_allocation(budget, scenario_cfg):
    """
    LP phân bổ tối ưu 4 hạng mục.
    
    Args:
        budget: Ngân sách đầu tư
        scenario_cfg: Dictionary cấu hình kịch bản
        
    Returns:
        dict: Chứa phân bổ I, D, AI, H
    """
    w = scenario_cfg['alloc_weights']
    c = [-w[0], -w[1], -w[2], -w[3]]
    A_ub = [[1, 1, 1, 1]]
    b_ub = [float(budget)]
    mins = [budget*0.10, budget*0.08, budget*0.08, budget*0.08]
    bounds = [(mins[i], budget*0.55) for i in range(4)]
    
    res = linprog(c, A_ub=A_ub, b_ub=b_ub, bounds=bounds, method='highs')
    
    if res.success:
        x = res.x
    else:
        x = np.array(w) * budget
        
    return {
        'I':  round(float(x[0]), 2),
        'D':  round(float(x[1]), 2),
        'AI': round(float(x[2]), 2),
        'H':  round(float(x[3]), 2),
    }

In [ ]:
def solve_m3_priority(scenario_cfg):
    """
    Xếp hạng ưu tiên ngành.
    
    Args:
        scenario_cfg: Dictionary cấu hình kịch bản
        
    Returns:
        list: Danh sách các ngành đã xếp hạng
    """
    data = get_data()
    X = data.X_sectors
    SECTORS = data.sectors_names_vi.tolist()
    
    good, bad = X[:, :6], X[:, 6]
    Gn = (good - good.min(0)) / (np.ptp(good, axis=0) + 1e-9)
    R  = (bad - bad.min()) / (np.ptp(bad) + 1e-9)
    
    w  = np.array([0.15, 0.15, 0.20, 0.15, 0.10, 0.20])
    w /= w.sum()
    
    score = Gn @ w - 0.15 * R
    idx   = np.argsort(-score)
    
    return [{'sector': SECTORS[i], 'score': round(float(score[i]), 4)} for i in idx]

In [ ]:
def solve_m4_labor(scenario_cfg):
    """
    Tác động lao động AI.
    
    Args:
        scenario_cfg: Dictionary cấu hình kịch bản
        
    Returns:
        dict: Chứa thông tin về dịch chuyển việc làm
    """
    data = get_data()
    EMPLOYMENT = data.sectors_employment
    AUTOMATION_RISK = data.sectors_automation_risk
    SECTORS = data.sectors_names_vi.tolist()
    
    ai_rate = scenario_cfg['ai_adoption']
    jobs_lost    = EMPLOYMENT * AUTOMATION_RISK * ai_rate
    jobs_created = jobs_lost * 0.4
    net          = jobs_lost - jobs_created
    
    return {
        'sectors':  SECTORS,
        'net_jobs': [round(float(-v), 3) for v in net],  # âm = mất việc
        'total_lost':    round(float(jobs_lost.sum()), 2),
        'total_created': round(float(jobs_created.sum()), 2),
        'net_total':     round(float(-net.sum()), 2),
    }

In [ ]:
def solve_m5_topsis(scenario_cfg):
    """
    TOPSIS xếp hạng vùng.
    
    Args:
        scenario_cfg: Dictionary cấu hình kịch bản
        
    Returns:
        list: Danh sách các vùng đã xếp hạng
    """
    data = get_data()
    regions = data.regions_names_vi.tolist()
    X = data.X_regions
    is_benefit = [True, True, True, True, True, False]
    
    norm = np.sqrt((X**2).sum(axis=0))
    R = X / (norm + 1e-9)
    
    # entropy weights
    P = R / (R.sum(axis=0) + 1e-9)
    P = np.clip(P, 1e-9, 1)
    E = -1/np.log(6) * np.sum(P * np.log(P), axis=0)
    w = (1 - E) / ((1 - E).sum() + 1e-9)
    V = R * w
    
    ideal      = np.array([V[:, j].max() if is_benefit[j] else V[:, j].min() for j in range(6)])
    anti_ideal = np.array([V[:, j].min() if is_benefit[j] else V[:, j].max() for j in range(6)])
    
    D_plus  = np.sqrt(((V - ideal)**2).sum(axis=1))
    D_minus = np.sqrt(((V - anti_ideal)**2).sum(axis=1))
    C = D_minus / (D_plus + D_minus + 1e-9)
    idx = np.argsort(-C)
    
    return [{'region': regions[i], 'score': round(float(C[i]), 4), 'rank': int(np.where(idx == i)[0][0]+1)} for i in range(6)]

In [ ]:
def solve_m6_risk(allocation, gdp_forecast, scenario_cfg):
    """
    Đánh giá rủi ro tổng hợp.
    
    Args:
        allocation: Phân bổ ngân sách
        gdp_forecast: Dự báo GDP
        scenario_cfg: Cấu hình kịch bản
        
    Returns:
        dict: Chứa thông tin về rủi ro
    """
    ai_share = allocation['AI'] / max(sum(allocation.values()), 1)
    gdp_growth = (gdp_forecast['gdp'][-1] / max(gdp_forecast['gdp'][0], 1) - 1)
    
    risk_score = float(np.clip(0.6 - gdp_growth * 0.5 - ai_share * 0.3, 0.1, 0.9))
    level = 'Thấp' if risk_score < 0.35 else 'Trung bình' if risk_score < 0.65 else 'Cao'
    
    return {
        'risk_score': round(risk_score, 3),
        'level': level,
        'gdp_growth_5y': round(gdp_growth * 100, 2),
        'ai_budget_share': round(ai_share * 100, 2),
    }

In [ ]:
def solve_bai12(budget=100, w_gdp=0.35, w_equity=0.30, w_ai=0.25, risk_threshold=0.55):
    """Alias pipeline cho bài 12 (dùng trong test)."""
    return solve_bai12_dashboard(total_budget=budget * 1000, scenario='S5')

In [ ]:
def solve_bai12_dashboard(data_dir=None, total_budget=50000, scenario='S5'):
    """
    Dashboard tổng hợp AIDEOM-VN với 5 kịch bản chính sách.

    Args:
        data_dir: Đường dẫn data (không bắt buộc)
        total_budget: Ngân sách tổng (Tỷ VND hoặc nghìn tỷ tuỳ app)
        scenario: 'S1' đến 'S5'

    Returns:
        dict với 6 module kết quả
    """
    cfg = SCENARIOS.get(scenario, SCENARIOS['S5'])

    # Chuẩn hóa budget về đơn vị nghìn tỷ cho tính toán
    budget_k = total_budget / 1000 if total_budget > 500 else float(total_budget)

    # Chạy 6 module
    allocation   = solve_m2_allocation(budget_k, cfg)
    gdp_forecast = solve_m1_macro(budget_k, cfg)
    priority     = solve_m3_priority(cfg)
    labor_impact = solve_m4_labor(cfg)
    topsis       = solve_m5_topsis(cfg)
    risk         = solve_m6_risk(allocation, gdp_forecast, cfg)

    return {
        'scenario':     scenario,
        'scenario_name': cfg['desc'],
        'description':  cfg['desc'],
        'budget':       total_budget,

        # Module 1 - GDP forecast
        'gdp_forecast': gdp_forecast,

        # Module 2 - Budget allocation
        'allocation': allocation,

        # Module 3 - Sector priority
        'priority': priority,

        # Module 4 - Labor impact
        'labor_impact': labor_impact,

        # Module 5 - TOPSIS region ranking
        'topsis': topsis,

        # Module 6 - Risk assessment
        'risk': risk,

        # Radar data tổng hợp
        'radar': {
            'dimensions': ['GDP', 'Hạ tầng', 'AI', 'Công bằng', 'Nhân lực', 'Rủi ro thấp'],
            'values': [
                min(95, max(20, gdp_forecast['gdp'][-1] / 20 * 100)),
                min(95, max(20, allocation['I'] / budget_k * 100 * 2)),
                min(95, max(20, allocation['AI'] / budget_k * 100 * 3)),
                min(95, max(20, allocation['H'] / budget_k * 100 * 3)),
                min(95, max(20, allocation['D'] / budget_k * 100 * 2.5)),
                min(95, max(20, (1 - risk['risk_score']) * 100)),
            ]
        },

        # Scenario comparison
        'all_scenarios': {
            sc: SCENARIOS[sc]['desc'] for sc in SCENARIOS
        },
    }

In [ ]:
if __name__ == '__main__':
    res = solve_bai12_dashboard()
    # In ra một số key để kiểm tra
    if isinstance(res, dict):
        print(res.keys())